## Churn Analysis and Customer Intelligence

In [227]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

## 1. Import database / data  

In [228]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("customer_churn.db")

df_db_customer = pd.read_sql(
    "SELECT * FROM Customers",
    conn
)

df_db_subscription = pd.read_sql(
    "SELECT * FROM Subscriptions",
    conn
)

df_db_support = pd.read_sql(
    "SELECT * FROM Support",
    conn
)

conn.close()

print("Customer:", df_db_customer.shape)
print("Subscription:", df_db_subscription.shape)
print("Support:", df_db_support.shape)

Customer: (5021, 8)
Subscription: (5021, 11)
Support: (2435, 6)


In [229]:
# Print table names and column names
conn = sqlite3.connect('customer_churn.db')

for table_name in tables['name']:
    print(f"\nTable Name: {table_name}")
    # Get column information
    columns_query = f"PRAGMA table_info({table_name});"
    columns = pd.read_sql(columns_query, conn)
    print("Columns:")
    print(columns['name'].tolist())

# Close connection
conn.close()


Table Name: Customers
Columns:
['customerid', 'name', 'country', 'state', 'gender', 'dob', 'interests', 'pincode']

Table Name: Subscriptions
Columns:
['customerid', 'subscription_start_date', 'subscription_type', 'renewal_date', 'plan_type', 'contract_type', 'cancellation_date', 'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score']

Table Name: Support
Columns:
['customerid', 'complaint_date', 'escalations', 'csat_score', 'col_1', 'comment']


## 2. Data cleaning

In [230]:
# df_db_Customers.head() # customer table
df_db_customer.tail()

,customerid,name,country,state,gender,dob,interests,pincode
5016,1333-LTHII,Diya,India,Madhya Pradesh,Female,2000-08-11,movie,643860.0
5017,9869-IUZDT,Aarav,India,Madhya Pradesh,Male,1987-09-04,sports,751764.0
5018,4004-ZHQJV,Tanvi,India,Rajasthan,Female,1969-03-12,movie,932073.0
5019,7367-YXSAV,Kavya,India,Rajasthan,Male,1996-02-17,technology,823598.0
5020,8115-AKLMU,Sakshi,India,Delhi,Female,1982-03-13,reading,429633.0


In [231]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5021 entries, 0 to 5020
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   customerid  5021 non-null   object 
 1   name        5021 non-null   object 
 2   country     5018 non-null   object 
 3   state       5021 non-null   object 
 4   gender      5021 non-null   object 
 5   dob         5021 non-null   object 
 6   interests   4509 non-null   object 
 7   pincode     5000 non-null   float64
dtypes: float64(1), object(7)
memory usage: 313.9+ KB


In [232]:
# a. rename col - name to customer_name
# b. drop columns - interest and pincode
# c. change data type - dob
# d. data standardization - gender
# e. fix missing values (using existing data) - country

#### a. Rename Col

In [233]:
# a. rename col - name to customer_name

df_db_customer.rename(columns = {'name' : 'customer_name'}, inplace= True)

#### b. Drop columns

In [234]:
df_db_customer.drop(columns=['interests', 'pincode'], inplace=True)  # using col name

In [235]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5021 entries, 0 to 5020
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customerid     5021 non-null   object
 1   customer_name  5021 non-null   object
 2   country        5018 non-null   object
 3   state          5021 non-null   object
 4   gender         5021 non-null   object
 5   dob            5021 non-null   object
dtypes: object(6)
memory usage: 235.5+ KB


#### c. Change data type

In [236]:
# c. change data type - dob

df_db_customer['dob'] = pd.to_datetime(df_db_customer['dob'])

#### d. Data standardization

In [237]:
# d. data standardization - gender

# df_db_customer['gender'].unique()
df_db_customer['gender'] = df_db_customer['gender'].replace({'Men' : 'Male', 'Women' : 'Female'})

In [238]:
df_db_customer['gender'].unique()

array(['Male', 'Female'], dtype=object)

#### e. Fix missing values

In [239]:
# e. fix missing values - country

# df_db_customer['country'].isna().sum()
df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,None,Delhi,Female,1988-12-10
8,0015-UOCOJ,maya,None,Kathmandu,Female,1985-07-07
12,0018-NYROU,chitra,None,Telangana,Female,2004-12-01


In [240]:
# country and state - unique value pair

# Creating state → country map from non-null rows
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

# Fill the missing country using State
df_db_customer['country'] = df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [241]:
df_db_customer[df_db_customer['country'].isna()] # no null value in country col

,customerid,customer_name,country,state,gender,dob


In [242]:
df_db_subscription.head() # subcription table

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,None,None,13.99,627.0,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150.0,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,None,None,6.99,210.0,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,None,None,22.99,1725.0,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195.0,88


In [243]:
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5021 entries, 0 to 5020
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               5021 non-null   object 
 1   subscription_start_date  5021 non-null   object 
 2   subscription_type        5021 non-null   object 
 3   renewal_date             5021 non-null   object 
 4   plan_type                5021 non-null   object 
 5   contract_type            5021 non-null   object 
 6   cancellation_date        1540 non-null   object 
 7   cancellation_reason      1540 non-null   object 
 8   monthly_charges          5021 non-null   float64
 9   cltv                     5021 non-null   float64
 10  churn_score              5021 non-null   int64  
dtypes: float64(2), int64(1), object(8)
memory usage: 431.6+ KB


In [244]:
# change data type to date - subscription_start_date , renewal_date, cancellation_date
date_col = ['subscription_start_date', 'renewal_date', 'cancellation_date']

df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5021 entries, 0 to 5020
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               5021 non-null   object        
 1   subscription_start_date  5021 non-null   datetime64[ns]
 2   subscription_type        5021 non-null   object        
 3   renewal_date             5021 non-null   datetime64[ns]
 4   plan_type                5021 non-null   object        
 5   contract_type            5021 non-null   object        
 6   cancellation_date        1540 non-null   datetime64[ns]
 7   cancellation_reason      1540 non-null   object        
 8   monthly_charges          5021 non-null   float64       
 9   cltv                     5021 non-null   float64       
 10  churn_score              5021 non-null   int64         
dtypes: datetime64[ns](3), float64(2), int64(1), object(5)
memory usage: 431.6+ KB


In [245]:
df_db_support.head() # support table

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28,N,60,None,service issue
1,0003-MKNFE,2024-08-28,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20,Y,20,None,None
3,0013-MHZWF,2025-03-18,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01,N,30,None,None


In [246]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2435 entries, 0 to 2434
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      2435 non-null   object
 1   complaint_date  2435 non-null   object
 2   escalations     2435 non-null   object
 3   csat_score      2435 non-null   int64 
 4   col_1           0 non-null      object
 5   comment         2430 non-null   object
dtypes: int64(1), object(5)
memory usage: 114.3+ KB


In [247]:
# drop columns
df_db_support.drop(columns=['col_1', 'comment'], inplace=True)

In [248]:
# change data type to date - complaint_date

df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2435 entries, 0 to 2434
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   customerid      2435 non-null   object        
 1   complaint_date  2435 non-null   datetime64[ns]
 2   escalations     2435 non-null   object        
 3   csat_score      2435 non-null   int64         
dtypes: datetime64[ns](1), int64(1), object(2)
memory usage: 76.2+ KB


## 3. Feature Engineering & Data Analysis

#### Create a new col

In [249]:
# create a new col using existing col - churn flag

# Customer is churned if cancellation_date is not null
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(), 1, 0)
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score,churn_flag
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627.0,12,0
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150.0,91,1
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,None,6.99,210.0,34,0
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,None,22.99,1725.0,8,0
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195.0,88,1


#### Merge Dataframes - JOINs

In [250]:
# first fix support table duplicates then merge
df = (df_db_subscription
            .merge(df_db_customer, on = 'customerid', how= 'left')
            .merge(df_db_support, on = 'customerid', how= 'left') )

In [251]:
df_db_subscription.shape

(5021, 12)

In [252]:
df.shape

(5582, 20)

In [253]:
print('df_db_subscription unique value:', df_db_subscription['customerid'].nunique())
print('df_db_customer unique value:', df_db_customer['customerid'].nunique())
print('df_db_support unique value:', df_db_support['customerid'].nunique())
print('df_db_support all value:', df_db_support['customerid'].size)

df_db_subscription unique value: 5021
df_db_customer unique value: 5021
df_db_support unique value: 1874
df_db_support all value: 2435


In [254]:
df_db_support

,customerid,complaint_date,escalations,csat_score
0,0003-MKNFE,2024-08-28,N,60
1,0003-MKNFE,2024-08-28,Y,10
2,0013-EXCHZ,2024-01-20,Y,20
3,0013-MHZWF,2025-03-18,N,90
4,0013-SMEOE,2024-11-01,N,30
...,...,...,...,...
2430,9931-PXWWG,2023-02-18,N,60
2431,0783-VIILJ,2024-10-05,N,70
2432,4770-BIGDV,2020-06-29,N,70
2433,8115-AKLMU,2024-02-08,Y,40


In [255]:
df_db_support['complaint_count'] = df_db_support.groupby('customerid')['customerid'].transform('count')

In [256]:
df_db_support = df_db_support.sort_values('complaint_date').drop_duplicates('customerid', keep= 'last')

In [257]:
df_db_support['customerid'].size

1874

In [258]:
# merge df
df = (df_db_subscription
            .merge(df_db_customer, on = 'customerid', how= 'left')
            .merge(df_db_support, on = 'customerid', how= 'left') )

In [259]:
df.shape

(5021, 21)

#### Export dataframe

In [265]:
df.to_csv('exported_clean_churn_data.csv', index=False)
df.to_excel('exported_clean_churn_data.xlsx', index=False)

## Data Analysis

In [263]:
# 1. Churn Rate

In [264]:
churn_rate = df['churn_flag'].mean()*100
print("Churn Rate = ", round(churn_rate,2), "%")

Churn Rate =  30.67 %


In [ ]:
# 2. Retenion Rate
retention_rate = 100 - churn_rate
print("Retention Rate = ", round(retention_rate,2), "%")

In [ ]:
# 3. Churn by Plan type
churn_by_plan = (df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name='churn_rate_pct'))
print(churn_by_plan)

In [ ]:
# 5. ARPU - Avg Revenue per user
arpu = df['monthly_charges'].mean()
print('ARPU = ', round(arpu,2))

In [ ]:
# 6. Avg Customer Tenure
# count of days users has used our service : cancellation date else current date
today = pd.Timestamp.today()

df['tenure_days'] = np.where(
        df['cancellation_date'].notna(),

    (df['cancellation_date'] - df['subscription_start_date']).dt.days,

    (today - df['subscription_start_date']).dt.days
)

avg_tenure = df['tenure_days'].mean()
print("Avg Tenure (Days) = ", round(avg_tenure),0)

In [ ]:
# 7. Revenue at risk - revenue lost from churned users
revenue_at_risk = df.loc[df['churn_flag']==1, 'monthly_charges'].sum()
print("Revenue at Risk (Rs 'K') =", revenue_at_risk)

In [ ]:
# 8. Esclation Rate
escalation_rate = (df['escalations']=='Y').mean()*100
print("Esclation Rate = ", round(escalation_rate, 2), "%")

In [ ]:
# 9. Avg Complaint Per User
avg_complaints = df['complaint_count'].sum() / df['customerid'].nunique()
print("Avg Compliants Per User = ", round(avg_complaints, 2))

In [ ]:
# 10. Correlation Esclation vs Churn
df['escalations'] = np.where(df['escalations'] == 'Y', 1, 0) # encoding string to int type
corr_df = df[['escalations', 'churn_flag']].dropna()

correlation = corr_df['escalations'].corr(df['churn_flag'])
print("Correlation between esclation vs churn is = ", round(correlation,2))

In [ ]:
# 11. Create a column using existing col - Churn risk
conditions = [
        (df['churn_score'] < 50),
        (df['churn_score'] >= 50) & (df['churn_score'] < 70),
        (df['churn_score'] >= 70)
]

choices = ['low', 'med', 'high']

df['churn_risk'] = np.select(conditions, choices, default='unkown')

## 4. Visualization using Matplotlib

In [ ]:
df_visual = df.copy()

In [ ]:
df_visual.columns

In [ ]:
# 4.1 Monthly Churn Trend (Time Series KPI)

df_visual['cancellation_month'] = df_visual['cancellation_date'].dt.to_period('M')

churn_trend = df_visual[df_visual['churn_flag'] == 1].groupby('cancellation_month').size()

plt.figure(figsize=(8,3))
plt.plot(churn_trend.index.astype(str), churn_trend.values,  color='green', marker='o', linestyle='dashed',  linewidth=2, markersize=12)

plt.title('Monthly Churn Trend')
plt.xlabel('Month')
plt.ylabel('Churned Customers')
plt.show()

In [ ]:
# 4.2 Churn Rate by Plan type
churn_plan = df_visual.groupby('plan_type')['churn_flag'].mean()

# colors = ['yellow', 'purple', 'blue']
colors = plt.cm.Set2(np.linspace(0, 1, len(churn_plan)))

plt.figure(figsize=(7,4))
plt.bar(churn_plan.index, churn_plan.values, color = colors)

plt.title('Churn Rate by Plan Type')
plt.xlabel('Plan Type')
plt.ylabel('Churn Rate (%)')
plt.show()

In [ ]:
# 4.3 Churn by States
churn_plan = df_visual.groupby('state')['churn_flag'].mean()

# colors = ['yellow', 'purple', 'blue']
colors = plt.cm.Set2(np.linspace(0, 1, len(churn_plan)))

plt.figure(figsize=(12,4))
plt.bar(churn_plan.index, churn_plan.values, color = colors)

plt.title('Churn Rate by Country')
plt.xlabel('State')
plt.ylabel('Churn Rate (%)')
plt.xticks(rotation=45)
plt.show()

## Visulaization using Seaborn

In [ ]:
# Heatmap (Correlation Matrix)

In [ ]:
# encoding - convert str to numeric so that we can find corr between features
df_visual.columns

In [ ]:
df_visual[['plan_type', 'contract_type', 'churn_score', 'churn_flag', 'churn_risk', 'escalations']].head()

In [ ]:
# remove warnings
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df_encoded = df_visual[['plan_type', 'contract_type', 'churn_score', 'churn_flag', 'churn_risk', 'escalations']]

categorial_cols = ['plan_type', 'contract_type', 'churn_risk']

for col in categorial_cols:
    df_encoded[col] = df_encoded[col].astype('category').cat.codes

In [ ]:
# Heatmap (correlation matrix)

sns.heatmap(df_encoded.corr(), annot=True)

In [ ]:
print('plan_type :', df_visual['plan_type'].unique())
print('contract_type :', df_visual['contract_type'].unique())
print('churn_risk :', df_visual['churn_risk'].unique())

In [ ]:
# Correct method of encoding - based on priority
df_encoded = df_visual[['plan_type', 'contract_type', 'churn_score', 'churn_flag', 'churn_risk', 'escalations']]

order_mappings = {
    'plan_type' : ['Basic', 'Standard', 'Premium'],
    'contract_type' : ['Monthly', 'Annual'],
    'churn_risk': ['low', 'med', 'high']
    }

for col, order in order_mappings.items():
    df_encoded[col] = pd.Categorical(df_encoded[col].astype('category'), categories=order, ordered=True).codes

In [ ]:
df_visual[['plan_type', 'contract_type', 'churn_score', 'churn_flag', 'churn_risk', 'escalations']].head()

In [ ]:
df_encoded.head()

In [ ]:
# Heatmap (correlation matrix)
sns.heatmap(df_encoded.corr(), annot=True)

In [ ]:
# Heatmap Using Matplotlib

corr_matrix = df_encoded.corr() # Correlation matrix

fig, ax = plt.subplots(figsize=(10, 6)) # Create figure

cax = ax.imshow(corr_matrix, cmap='coolwarm') # Heatmap

fig.colorbar(cax) # Add colorbar

# Axis labels
ax.set_xticks(np.arange(len(corr_matrix.columns)))
ax.set_yticks(np.arange(len(corr_matrix.columns)))

ax.set_xticklabels(corr_matrix.columns, rotation=45)
ax.set_yticklabels(corr_matrix.columns)

# Annotate values inside cells
for i in range(len(corr_matrix.columns)):
    for j in range(len(corr_matrix.columns)):
        ax.text(
            j,
            i,
            f"{corr_matrix.iloc[i, j]:.2f}",
            ha='center',
            va='center'
        )

plt.title('Correlation Heatmap') # Title

plt.tight_layout()
plt.show()

In [ ]:
# pairplot - Plot pairwise relationships in a dataset
sns.pairplot(df_encoded)

In [ ]:
df_visual.columns

In [ ]:
# catplt/Facegrid plot - multi-dim comparison

sns.catplot(data=df_visual,
    x='plan_type',
    y='monthly_charges',
    hue='gender',
    col='churn_risk')

## Pivot table

In [ ]:
# Pivot Table

pd.pivot_table(
    df_visual,
    index='plan_type',
    values='churn_flag',
    aggfunc = 'mean'
)

In [266]:
# pivot table using multiple cols and agg type

pd.pivot_table(
    df_visual,
    index='plan_type',
    values=['monthly_charges', 'customerid', 'churn_flag'],
    aggfunc = {
        'monthly_charges' : 'sum',
        'customerid' : 'nunique',
        'churn_flag' : 'mean'
    }
)

,churn_flag,customerid,monthly_charges
plan_type,,,
Basic,0.318126,1622,12301.78
Premium,0.297683,1683,39723.17
Standard,0.304779,1716,25063.84


In [267]:
import os

for file in os.listdir("/content"):
    print(file)

.config
customer_churn.db
.ipynb_checkpoints
customer_churn_data.xlsx
exported_clean_churn_data.xlsx
exported_clean_churn_data.csv
sample_data


In [268]:
!zip -r /content/Customer_Churn_Project_All_Files.zip /content/*

  adding: content/customer_churn_data.xlsx (deflated 2%)
  adding: content/customer_churn.db (deflated 64%)
  adding: content/exported_clean_churn_data.csv (deflated 74%)
  adding: content/exported_clean_churn_data.xlsx (deflated 3%)
  adding: content/sample_data/ (stored 0%)
  adding: content/sample_data/README.md (deflated 39%)
  adding: content/sample_data/anscombe.json (deflated 83%)
  adding: content/sample_data/california_housing_test.csv (deflated 76%)
  adding: content/sample_data/california_housing_train.csv (deflated 79%)
  adding: content/sample_data/mnist_test.csv (deflated 88%)
  adding: content/sample_data/mnist_train_small.csv (deflated 88%)
